In [1]:
import pandas as pd
import plotly.express as px

# Recuerda cambiar 'tus_datos.csv' por el nombre real de tu archivo
df = pd.read_csv('../ventas Google jun-1-2026.csv') 

# Para ver los nombres de las columnas y saber qué graficar
df.head()

,Día,Impr.,Clics,Pedidos,Código de moneda,Coste,Valor conv./coste
0,2023-01-01,403.000,29.0,0,COP,31868,0
1,2023-01-02,10.817,171.0,0,COP,32130,0
2,2023-01-03,14.489,302.0,0,COP,"66562,29",0
3,2023-01-04,20.518,644.0,0,COP,"113332,28",0
4,2023-01-05,16.059,401.0,0,COP,"77880,63",0


In [3]:
# Cambia 'nombre_de_columna_numerica' por una columna real de tu dataset
fig_hist = px.histogram(df, x="Impr.", title="Distribución de mi variable")
fig_hist.show()

In [5]:
# Cambia 'columna_X' y 'columna_Y' por dos variables numéricas de tu tabla
fig_scatter = px.scatter(df, x="Impr.", y="Clics", title="Relación entre X y Y")
fig_scatter.show()

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# 1. Cargar el dataset original pesado
# Intento: usar archivo esperado, luego variable en memoria ('df'), luego cualquier CSV disponible
candidates = ['archivo_limpio_final.csv']
df_sales = None
source = None

for path in candidates:
    if os.path.exists(path):
        df_sales = pd.read_csv(path, low_memory=False)
        source = path
        break

if df_sales is None:
    if 'df' in globals():
        df_sales = df.copy()
        source = "in-memory variable 'df'"
    else:
        csvs = glob.glob('*.csv') + glob.glob('../*.csv')
        if csvs:
            csvs.sort(key=os.path.getmtime, reverse=True)
            chosen = csvs[0]
            df_sales = pd.read_csv(chosen, low_memory=False)
            source = chosen

if df_sales is None:
    raise FileNotFoundError("No se encontró 'archivo_limpio_final.csv', tampoco existe la variable 'df' ni hay otros CSV en el directorio. Coloca el archivo o carga los datos en una celda previa.")

print(f"Loaded dataframe from: {source} (rows={len(df_sales)})")

# 2. Aplicar los filtros de negocio estrictos que ya tenías
df_sales = df_sales[df_sales['Status raw value (temporary)'].str.lower() == 'invoiced'].drop_duplicates(subset=['Order'], keep='first')

# 3. Formatear fechas y extraer componentes cronológicos
df_sales['Creation Date'] = pd.to_datetime(df_sales['Creation Date'])
df_sales['Year'] = df_sales['Creation Date'].dt.year
df_sales['Month_Num'] = df_sales['Creation Date'].dt.month

# 4. Determinar Clientes Nuevos vs Recurrentes
df_sales['First_Purchase_Date'] = df_sales.groupby('Client Document')['Creation Date'].transform('min')
df_sales['Tipo_Cliente'] = np.where(df_sales['Creation Date'] == df_sales['First_Purchase_Date'], 'Nuevo', 'Recurrente')

# 5. Atribución de Canales (UTMs)
df_sales['UtmMedium'] = df_sales['UtmMedium'].astype(str).str.lower().str.strip()
palabras_pauta_reales = ['cpc', 'cpa', 'cpa+', 'paid', 'facebook', 'conversion', 'trafico']
df_sales['Origen_Orden'] = np.where(
    df_sales['UtmMedium'].isin(['nan', '', 'none', 'null']), 'Orgánico',
    np.where(df_sales['UtmMedium'].str.contains('|'.join(palabras_pauta_reales)), 'Anuncios (Pauta)', 'Orgánico')
)

# 6. Filtrar ÚNICAMENTE el rango necesario para los gráficos (Ene - Jun 2026)
df_2026 = df_sales[(df_sales['Year'] == 2026) & (df_sales['Month_Num'] <= 6)].copy()
meses_espanol = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio'}
df_2026['Mes'] = df_2026['Month_Num'].map(meses_espanol)

# 7. SELECCIONAR EXCLUSIVAMENTE LAS COLUMNAS REQUERIDAS (Esto elimina el exceso de megabytes)
columnas_necesarias = ['Mes', 'Creation Date', 'Tipo_Cliente', 'Client Document', 'Order']
df_web_optimized = df_2026[columnas_necesarias].copy()

# 8. Guardar en un archivo compacto optimizado
df_web_optimized.to_csv('datos_web_dashboard.csv', index=False)
print("¡Archivo 'datos_web_dashboard.csv' generado con éxito y optimizado para la web!")

FileNotFoundError: [Errno 2] No such file or directory: 'archivo_limpio_final.csv'